# ML-04 — Search Intelligence Data Contract

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [28]:
from dotenv import load_dotenv
import os

load_dotenv()
print("Token loaded:", "HF_TOKEN" in os.environ)

Token loaded: True


In [29]:
from huggingface_hub import list_repo_files
import os

files = list_repo_files("FlyRank/internship-warehouse", repo_type="dataset", token=os.environ["HF_TOKEN"])
for f in files[:30]:
    print(f)

.gitattributes
README.md
dim_clients.parquet
dim_content.parquet
fact_content_daily_performance/month=2025-01/data_0.parquet
fact_content_daily_performance/month=2025-02/data_0.parquet
fact_content_daily_performance/month=2025-03/data_0.parquet
fact_content_daily_performance/month=2025-04/data_0.parquet
fact_content_daily_performance/month=2025-05/data_0.parquet
fact_content_daily_performance/month=2025-06/data_0.parquet
fact_content_daily_performance/month=2025-07/data_0.parquet
fact_content_daily_performance/month=2025-08/data_0.parquet
fact_content_daily_performance/month=2025-09/data_0.parquet
fact_content_daily_performance/month=2025-10/data_0.parquet
fact_content_daily_performance/month=2025-11/data_0.parquet
fact_content_daily_performance/month=2025-12/data_0.parquet
fact_content_daily_performance/month=2026-01/data_0.parquet
fact_content_daily_performance/month=2026-02/data_0.parquet
fact_content_daily_performance/month=2026-03/data_0.parquet
fact_content_daily_performance/mont

In [30]:
import duckdb
import os

con = duckdb.connect()
con.sql("INSTALL httpfs; LOAD httpfs;")

con.sql(f"""
CREATE OR REPLACE SECRET hf_token (
    TYPE huggingface,
    TOKEN '{os.environ["HF_TOKEN"]}'
);
""")

print("DuckDB connected, token set")

DuckDB connected, token set


## 1. Unit of analysis + time window

**One row = one content page, for one client, on one specific day** (a page-client-day combination), 
in the `fact_content_daily_performance` table.

**Table(s) used:** `fact_content_daily_performance` (daily facts) 

**Time window:** the mid-panel month `month=2026-03` (March 1–31, 2026) for developing features and 
label logic. The final month (`2026-06`, and the `_sample` table which is exactly June 2026) is 
treated as a sealed test month — never used to build label logic, since it's the natural "future" 
outcome window for any past→future prediction.

**What I'd predict/rank:** whether a page is declining (or will decline) — a label built from 
observed signals over this window, used to rank pages for review priority (same as my Lane 2 
framing from w01/w02).

**One thing I deliberately exclude:** raw `fact_content_daily_performance_sample.parquet` for 
building labels — I verified it below is exactly June 2026, so using it now would leak the outcome 
window into my feature/label design.

## 2. Fields: feature / label / context / excluded

**Features** (known at the decision moment, from the prior/current window):
- `gsc_impressions`, `gsc_clicks`, `gsc_avg_position`, `ctr` (calculated) — daily search performance signals
- `ga4_data_available` — flags whether engagement data exists for that row (used as a filter/context, 
  not a predictive feature on its own)

**Label/proxy:**
- A "declining" indicator built from a persistent drop in impressions/clicks/position across the 
  window — not yet finalized, will follow the future-window pattern from w02 (prior 90 days → next 
  30 days decline).

**Context** (background, not fed to the model directly):
- `content_hash_id`, `client_hash_id`, `report_date` — join keys and grouping only, no predictive 
  meaning themselves.

**Excluded:**
- Any product-computed score (`health_score`, `priority_score`, `action_type`) — these are rule 
  outputs, not observed signals, and aren't even shipped in this data by design.
- The `_sample` table / June 2026 window — reserved as a sealed test set, never used for feature or 
  label design (per the assignment's leakage warning).

## 3. Verify it with queries (grain, counts, missing values, windows)

**Query 1 — Grain check:** confirms one row = one unique (content, client, date) combination.

In [31]:
grain_check = con.sql("""
    SELECT COUNT(*) AS total_rows,
           COUNT(DISTINCT (content_hash_id || client_hash_id || CAST(report_date AS VARCHAR))) AS unique_combos
    FROM 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/data_0.parquet'
""").df()

grain_check

,total_rows,unique_combos
0,9841378,9841378


Result: 9,841,378 total rows = 9,841,378 unique combinations — grain confirmed, no duplicate rows.

**Query 2 — Row count + date span:** confirms the slice covers exactly the intended month.

In [32]:
count_span = con.sql("""
    SELECT COUNT(*) AS row_count,
           MIN(report_date) AS min_date,
           MAX(report_date) AS max_date
    FROM 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/data_0.parquet'
""").df()

count_span

,row_count,min_date,max_date
0,9841378,2026-03-01,2026-03-31


Result: 9,841,378 rows spanning 2026-03-01 to 2026-03-31 — full March coverage, no leakage from 
outside the intended window.

**Query 3 — Availability check (`IS TRUE`):** confirms how much of the slice actually has GA4 
engagement data, since not every row has full instrumentation.

In [33]:
availability = con.sql("""
    SELECT COUNT(*) AS total_rows,
           COUNT(*) FILTER (WHERE ga4_data_available IS TRUE) AS ga4_available_rows
    FROM 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/data_0.parquet'
""").df()

availability

,total_rows,ga4_available_rows
0,9841378,413966


Result: only 413,966 of 9,841,378 rows (~4.2%) have `ga4_data_available IS TRUE`. This means most 
rows in this slice are search-only (GSC) rows — any feature relying on GA4 engagement data will only 
apply to a small minority of rows, so I'll need to handle this sparsity explicitly rather than 
assume GA4 signals are broadly available.

In [34]:
features_df = con.sql("""
    SELECT
        content_hash_id,
        client_hash_id,
        report_date,
        gsc_impressions,
        gsc_clicks,
        gsc_avg_position,
        CASE WHEN gsc_impressions > 0 
             THEN CAST(gsc_clicks AS DOUBLE) / gsc_impressions 
             ELSE 0 END AS ctr,
        ga4_data_available
    FROM 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/data_0.parquet'
    LIMIT 1000
""").df()

features_df.head(10)

,content_hash_id,client_hash_id,report_date,gsc_impressions,gsc_clicks,gsc_avg_position,ctr,ga4_data_available
0,content_b7e512995f79d5a6,client_73cda7b4e4f265ea,2026-03-01,20,0,3.350000,0.000000,<NA>
1,content_05597932fe4da067,client_73cda7b4e4f265ea,2026-03-01,1,0,0.000000,0.000000,<NA>
2,content_7a105f548d9c6916,client_73cda7b4e4f265ea,2026-03-01,125,1,4.928000,0.008000,<NA>
3,content_905aa32a0230694e,client_73cda7b4e4f265ea,2026-03-01,7,0,4.000000,0.000000,<NA>
4,content_a3ea9792f793ec72,client_73cda7b4e4f265ea,2026-03-01,11,0,2.272727,0.000000,<NA>
5,content_36c36abc7650d7af,client_73cda7b4e4f265ea,2026-03-01,239,1,7.347280,0.004184,<NA>
6,content_a7da352b73b02668,client_73cda7b4e4f265ea,2026-03-01,191,0,7.832461,0.000000,<NA>
7,content_05434271b257bb68,client_73cda7b4e4f265ea,2026-03-01,55,0,3.272727,0.000000,<NA>
8,content_d056587ff7faca0c,client_73cda7b4e4f265ea,2026-03-01,77,0,5.636364,0.000000,<NA>
9,content_bfd1e41c2af250c8,client_73cda7b4e4f265ea,2026-03-01,2,0,4.500000,0.000000,<NA>


**Five features (max), each with "available at decision moment?" check:**

1. **`gsc_impressions`** — direct daily Search Console measurement recorded for that day; no future 
   information involved.
2. **`gsc_clicks`** — same-day observed measurement from Search Console.
3. **`gsc_avg_position`** — same-day average ranking position, observed directly from Search 
   Console for that day.
4. **`ctr`** (calculated: `gsc_clicks / gsc_impressions`) — knowable the moment the day's Search 
   Console data lands; no future window involved.
5. **`ga4_data_available`** — a same-day flag showing whether GA4 engagement data exists for that 
   row; known instantly, used as context/filter rather than a raw predictive signal.

In [35]:
import numpy as np
from sklearn.linear_model import LogisticRegression

# Simple proxy label: is this row's position "poor" (>10)?
features_df["is_poor_position"] = (features_df["gsc_avg_position"] > 10).astype(int)

# THE TRAP: a leaked feature built directly FROM the label's source column
features_df["leaked_feature"] = features_df["gsc_avg_position"] * 1.0

X_leak = features_df[["leaked_feature"]].fillna(0)
y = features_df["is_poor_position"]

model_leak = LogisticRegression().fit(X_leak, y)
score_leak = model_leak.score(X_leak, y)
print("Score WITH leaked feature:", score_leak)

Score WITH leaked feature: 0.998


In [36]:
# Remove the leaked feature, use only honest, pre-decision features
X_honest = features_df[["gsc_impressions", "gsc_clicks", "ctr"]].fillna(0)

model_honest = LogisticRegression().fit(X_honest, y)
score_honest = model_honest.score(X_honest, y)
print("Score WITHOUT leaked feature (honest):", score_honest)

Score WITHOUT leaked feature (honest): 0.813


**The trap, performed live:** I added `leaked_feature`, which is literally built from the same 
column (`gsc_avg_position`) used to define my label `is_poor_position` (position > 10). The "model" 
trained with this leaked feature scored **0.998** — suspiciously close to perfect. That's the 
giveaway: a score this high on a real-world search-ranking problem almost always means the label is 
hiding inside a feature, not that the model discovered something genuinely predictive.

After removing the leaked feature and using only honest, same-day observed signals 
(`gsc_impressions`, `gsc_clicks`, `ctr`), the score dropped to **0.813** — still decent, but a 
believable, non-perfect number. This is the number I actually trust, because none of these features 
were derived from the label itself — they're all independently observed search signals available 
at the same decision moment as the label.

## 4. Data limits

**Unbalanced history:** clients have very different amounts of tracking history — some have 12+ 
months, others started recently. Before comparing across clients or building seasonality features, 
I'd need to check each client's `gsc_data_start` / `ga4_data_start` in `dim_clients`, otherwise "no 
data yet" could be mistaken for "no traffic."

**GA4 sparsity:** in my March 2026 slice, only ~4.2% of rows had `ga4_data_available IS TRUE` 
(413,966 of 9,841,378). This means almost all rows here are search-only (GSC) — any feature relying 
on GA4 engagement (sessions, scroll, engagement rate) will only apply to a small minority of pages, 
so I can't assume engagement signals are broadly available across my whole slice.

**Window overlap risk:** because I'm currently building same-day features and a same-day label 
(`is_poor_position` from that day's `gsc_avg_position`), this notebook's label is a proxy, not a 
future outcome — it doesn't yet respect the "prior window → future window" discipline my actual 
capstone label will need. I have to be careful in later weeks not to let feature and target windows 
overlap the way this quick demo intentionally did to illustrate leakage.

**This data can never tell me:** whether a refresh action *caused* any recovery (no causal 
experiment here), or anything about Google's actual ranking algorithm — only observed, 
correlational signals about what happened.

## Self-check

Before you submit, confirm each line honestly:
- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.